# 06_phase3_pathway_enrichment.ipynb
Phase 3 — Pathway Enrichment

**Critical design point (per Adrien's brief):** the background gene list must be "all genes detected in the relevant cell type" — NOT gseapy's default whole-genome background. Getting this wrong inflates results, since it's much easier for a gene list to look enriched against ~20,000 genes than against the ~4,000-16,000 genes actually expressed and testable in a given cell type.

Background used here: the full gene set from each DE comparison's results table (i.e. every gene that passed the `min_genes_expressed` filter in the pseudobulk DE notebook) — this is exactly the correct, cell-type-and-comparison-specific universe.

In [1]:
# ----------------------------
# Cell 1 — Imports and paths
# ----------------------------
import gc
import numpy as np
import pandas as pd
import gseapy as gp
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_DIR = Path(r"C:\Users\annam\Dissertation 2026")
DE_RESULTS_DIR = PROJECT_DIR / "results" / "phase3_pseudobulk_de"
RESULTS_DIR = PROJECT_DIR / "results" / "phase3_pathway_enrichment"
FIGURE_DIR = PROJECT_DIR / "figures" / "phase3_pathway_enrichment"

for d in [RESULTS_DIR, FIGURE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Setup complete")

Setup complete


In [2]:
# ----------------------------
# Cell 2 — Gene set library
# Using MSigDB Hallmark gene sets (50 well-curated, non-redundant pathways
# covering core biological processes) as the primary library — a sensible
# default breadth for a first pass. GO Biological Process available as an
# optional secondary library if more granular results are wanted later.
# ----------------------------
gene_set_library = "MSigDB_Hallmark_2020"
print(f"Using gene set library: {gene_set_library}")
print("(requires internet access — gseapy fetches this from Enrichr's servers)")

Using gene set library: MSigDB_Hallmark_2020
(requires internet access — gseapy fetches this from Enrichr's servers)


In [3]:
# ----------------------------
# Cell 3 — Discover all DE comparison result files to run enrichment on
# Only using the FULL results files (not _significant) here, since we need
# both the significant gene list (foreground) AND the full tested gene
# list (background) from the same file for consistency.
# ----------------------------
de_files = sorted([
    f for f in DE_RESULTS_DIR.glob("*.csv")
    if not f.name.endswith("_significant.csv")
    and "summary" not in f.name
])

print(f"Found {len(de_files)} DE comparison result files:")
for f in de_files:
    print(f"  {f.name}")

Found 41 DE comparison result files:
  GSE114725_DE_B_cells_tumor_vs_normal.csv
  GSE114725_DE_CD8_Effector_T_cells_tumor_vs_normal.csv
  GSE114725_DE_Macrophages_tumor_vs_normal.csv
  GSE114725_DE_NK_Cytotoxic_T_cells_tumor_vs_normal.csv
  GSE114725_DE_T_cells_tumor_vs_normal.csv
  GSE176078_DE_B_cells_HER2+_vs_ER+.csv
  GSE176078_DE_B_cells_TNBC_vs_ER+.csv
  GSE176078_DE_B_cells_TNBC_vs_HER2+.csv
  GSE176078_DE_CAFs_HER2+_vs_ER+.csv
  GSE176078_DE_CAFs_TNBC_vs_ER+.csv
  GSE176078_DE_CAFs_TNBC_vs_HER2+.csv
  GSE176078_DE_CD8_T_cells_HER2+_vs_ER+.csv
  GSE176078_DE_CD8_T_cells_TNBC_vs_ER+.csv
  GSE176078_DE_CD8_T_cells_TNBC_vs_HER2+.csv
  GSE176078_DE_Cycling_epithelial_HER2+_vs_ER+.csv
  GSE176078_DE_Cycling_epithelial_TNBC_vs_ER+.csv
  GSE176078_DE_Cycling_epithelial_TNBC_vs_HER2+.csv
  GSE176078_DE_Endothelial_cells_HER2+_vs_ER+.csv
  GSE176078_DE_Endothelial_cells_TNBC_vs_ER+.csv
  GSE176078_DE_Endothelial_cells_TNBC_vs_HER2+.csv
  GSE176078_DE_Epithelial_ambiguous_HER2+_vs_ER+.csv

In [4]:
# ----------------------------
# Cell 4 — Pathway enrichment function
# For each DE comparison: builds the correctly-scoped background (all
# genes tested in THAT comparison, i.e. the full results file's gene
# index), takes the FDR-significant genes as foreground, and runs
# over-representation analysis (Fisher's exact test) via gseapy.enrich
# (local mode, explicit background — NOT gseapy.enrichr's online mode,
# which uses a generic whole-genome background by default).
# ----------------------------
def run_pathway_enrichment(de_csv_path, gene_set_library, min_sig_genes=5):
    results_df = pd.read_csv(de_csv_path, index_col=0)

    # Background = every gene actually tested in this specific comparison
    background_genes = results_df.index.tolist()

    # Foreground = FDR-significant genes from this same comparison
    sig_genes = results_df[results_df["padj"] < 0.05].index.tolist()

    if len(sig_genes) < min_sig_genes:
        print(f"  SKIP {de_csv_path.stem}: only {len(sig_genes)} significant genes "
              f"(need >= {min_sig_genes} for meaningful enrichment)")
        return None

    try:
        enr = gp.enrich(
            gene_list=sig_genes,
            gene_sets=gene_set_library,
            background=background_genes,
            outdir=None,
        )
        pathway_results = enr.results.copy()
        pathway_results = pathway_results.sort_values("Adjusted P-value")

        n_sig_pathways = (pathway_results["Adjusted P-value"] < 0.05).sum()
        print(f"  {de_csv_path.stem}: {len(sig_genes)} input genes (background={len(background_genes)}) "
              f"-> {n_sig_pathways} significant pathways (adj p<0.05)")
        return pathway_results

    except Exception as e:
        print(f"  FAILED {de_csv_path.stem}: {type(e).__name__}: {e}")
        return None

print("Pathway enrichment function ready")

Pathway enrichment function ready


In [5]:
# ----------------------------
# Cell 5 — Run enrichment across all DE comparisons
# ----------------------------
all_pathway_results = {}

for de_file in de_files:
    result = run_pathway_enrichment(de_file, gene_set_library)
    if result is not None:
        all_pathway_results[de_file.stem] = result
        result.to_csv(RESULTS_DIR / f"{de_file.stem}_pathways.csv", index=False)
        result[result["Adjusted P-value"] < 0.05].to_csv(
            RESULTS_DIR / f"{de_file.stem}_pathways_significant.csv", index=False)
    gc.collect()

print(f"\nPathway enrichment complete: {len(all_pathway_results)} comparisons produced results")

  SKIP GSE114725_DE_B_cells_tumor_vs_normal: only 1 significant genes (need >= 5 for meaningful enrichment)
  SKIP GSE114725_DE_CD8_Effector_T_cells_tumor_vs_normal: only 0 significant genes (need >= 5 for meaningful enrichment)
  GSE114725_DE_Macrophages_tumor_vs_normal: 12 input genes (background=9267) -> 0 significant pathways (adj p<0.05)
  SKIP GSE114725_DE_NK_Cytotoxic_T_cells_tumor_vs_normal: only 1 significant genes (need >= 5 for meaningful enrichment)
  SKIP GSE114725_DE_T_cells_tumor_vs_normal: only 1 significant genes (need >= 5 for meaningful enrichment)
  SKIP GSE176078_DE_B_cells_HER2+_vs_ER+: only 0 significant genes (need >= 5 for meaningful enrichment)
  GSE176078_DE_B_cells_TNBC_vs_ER+: 79 input genes (background=5851) -> 10 significant pathways (adj p<0.05)
  GSE176078_DE_B_cells_TNBC_vs_HER2+: 162 input genes (background=3891) -> 12 significant pathways (adj p<0.05)
  SKIP GSE176078_DE_CAFs_HER2+_vs_ER+: only 0 significant genes (need >= 5 for meaningful enrichment

In [6]:
# ----------------------------
# Cell 6 — Summary across all comparisons
# ----------------------------
summary_rows = []
for name, result in all_pathway_results.items():
    n_sig = (result["Adjusted P-value"] < 0.05).sum()
    top_pathway = result.iloc[0]["Term"] if len(result) > 0 else None
    top_padj = result.iloc[0]["Adjusted P-value"] if len(result) > 0 else None
    summary_rows.append({
        "comparison": name,
        "n_significant_pathways": n_sig,
        "top_pathway": top_pathway,
        "top_pathway_adj_pvalue": top_padj,
    })

summary_df = pd.DataFrame(summary_rows).sort_values("n_significant_pathways", ascending=False)
summary_df.to_csv(RESULTS_DIR / "phase3_pathway_summary_all_comparisons.csv", index=False)
print(summary_df.to_string(index=False))

                                     comparison  n_significant_pathways                       top_pathway  top_pathway_adj_pvalue
    GSE176078_DE_Luminal_epithelial_TNBC_vs_ER+                      18                       E2F Targets            7.606686e-13
  GSE176078_DE_Epithelial_ambiguous_TNBC_vs_ER+                      14           Estrogen Response Early            8.895847e-07
             GSE176078_DE_B_cells_TNBC_vs_HER2+                      12     TNF-alpha Signaling via NF-kB            4.109425e-12
               GSE176078_DE_B_cells_TNBC_vs_ER+                      10     TNF-alpha Signaling via NF-kB            1.342610e-09
    GSE176078_DE_Cycling_epithelial_TNBC_vs_ER+                       9           Estrogen Response Early            8.440926e-05
         GSE176078_DE_CD8_T_cells_TNBC_vs_HER2+                       4         Interferon Alpha Response            4.731942e-03
           GSE176078_DE_CD8_T_cells_TNBC_vs_ER+                       4             Inflam

## Figures — Pathway dot plots and summary heatmap

In [7]:
# ----------------------------
# Cell 7 — Pathway dot plots, one per comparison with >=1 pathway result
# X-axis: -log10(adjusted p-value). Dot size: gene overlap count.
# Only comparisons that produced at least one row are plotted (empty
# results already skipped upstream).
# ----------------------------
import matplotlib.pyplot as plt
import numpy as np

def plot_pathway_dotplot(pathway_df, title, save_path, top_n=15):
    df = pathway_df.head(top_n).copy()
    if len(df) == 0:
        return
    df["neg_log10_adj_p"] = -np.log10(df["Adjusted P-value"].clip(lower=1e-300))
    df["n_genes"] = df["Genes"].apply(lambda g: len(str(g).split(";")))
    df = df.sort_values("neg_log10_adj_p")

    fig, ax = plt.subplots(figsize=(8, max(3, len(df) * 0.4)))
    sig_mask = df["Adjusted P-value"] < 0.05
    colors = np.where(sig_mask, "firebrick", "grey")

    ax.scatter(df["neg_log10_adj_p"], range(len(df)),
               s=df["n_genes"] * 30, c=colors, alpha=0.7, edgecolors="black", linewidth=0.5)
    ax.set_yticks(range(len(df)))
    ax.set_yticklabels(df["Term"], fontsize=9)
    ax.axvline(-np.log10(0.05), color="grey", linestyle="--", linewidth=0.8)
    ax.set_xlabel("-log10(adjusted p-value)")
    ax.set_title(title, fontsize=11)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close()

for name, pathway_df in all_pathway_results.items():
    plot_pathway_dotplot(
        pathway_df, name.replace("_", " "),
        FIGURE_DIR / f"{name}_pathway_dotplot.png"
    )

print(f"Pathway dot plots saved: {len(all_pathway_results)} figures")

Pathway dot plots saved: 22 figures


In [8]:
# ----------------------------
# Cell 8 — Summary heatmap: significant pathway count across all comparisons
# Gives a single at-a-glance figure showing which cell types/comparisons
# had the strongest pathway-level signal — useful as a results-chapter
# overview figure alongside the individual dot plots.
# ----------------------------
heatmap_data = summary_df.set_index("comparison")["n_significant_pathways"]
heatmap_data = heatmap_data.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, max(4, len(heatmap_data) * 0.3)))
colors = plt.cm.Reds(heatmap_data / max(heatmap_data.max(), 1))
ax.barh(range(len(heatmap_data)), heatmap_data.values, color=colors, edgecolor="black", linewidth=0.3)
ax.set_yticks(range(len(heatmap_data)))
ax.set_yticklabels([c.replace("_", " ") for c in heatmap_data.index], fontsize=7)
ax.set_xlabel("Number of significant pathways (adj p<0.05)")
ax.set_title("Pathway enrichment summary — all comparisons")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "phase3_pathway_summary_heatmap.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.close()

print("Summary heatmap saved: phase3_pathway_summary_heatmap.png")

Summary heatmap saved: phase3_pathway_summary_heatmap.png
